# GateKeep · notebook 09 · demo app on the scikit-learn user guide

Starts a small web page where you type a question and see the route, the retrieved passages, each gate decision (Laya or Gemma, with its time) and the answer.
It answers from the scikit-learn user guide only (BSD-3-Clause, (c) The scikit-learn developers); the book is set aside and never loaded.
**Attach:** `gatekeep_data` and **notebook 04's output**. **Settings:** GPU T4 · Internet On · secret `OLLAMA_API_KEY`.
Run every cell. The last cell prints a link ending in `gradio.live`; open it in your browser. It works for about 3 days and anyone with the link can use it (and spend your Ollama quota), so do not share it.

## 1 · code, packages, key

In [ ]:
import glob, os, shutil, subprocess
os.chdir("/kaggle/working")
if os.path.isdir("GateKeep"):
    subprocess.run(["git", "-C", "GateKeep", "pull"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/RishabhCodezZz/GateKeep.git"], check=True)
os.chdir("GateKeep")
os.environ["PYTHONPATH"] = "src"   # so `!python -m gatekeep.cli` finds the package
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" safetensors huggingface_hub scipy accelerate faiss-cpu sentence-transformers pymupdf pytest openai docling gradio


In [ ]:
!nvidia-smi -L

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["OLLAMA_API_KEY"] = UserSecretsClient().get_secret("OLLAMA_API_KEY")

## 2 · restore (cache and data from the fullest backup, models from notebook 04's output)

In [ ]:
import sys
sys.path.insert(0, "src")
from gatekeep.restore import restore

COLD_START_OK = False   # set True only if you really want to rebuild everything (costs ~11% of the monthly quota)
where = restore("/kaggle/input", ".")
if where:
    print("restored from", where)
    subprocess.run(["python", "-m", "gatekeep.cli", "purge_empty"], check=True)  # drop empty replies cached before the thinking fix
elif not COLD_START_OK:
    raise SystemExit("STOPPED: no backup found in /kaggle/input. Attach your backup (the dataset must contain the folders "
                     "data/, cache/, results/ or a gatekeep_backup.zip), or set COLD_START_OK = True to rebuild from scratch.")
else:
    print("no backup: cold start, the book is re-parsed and every model call is repeated")
for name in ("review.csv", "handwritten.csv"):
    found = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if found and not os.path.exists(name):
        shutil.copy(found[0], name)
    print(name, "->", "found" if os.path.exists(name) else "MISSING (add it to your dataset)")

In [ ]:
missing = [g for g in ("route", "grade", "grounded", "sufficient") if not os.path.exists(f"models/{g}/model.safetensors")]
assert not missing, (f"fine-tuned weights missing for {missing}. Attach notebook 04's OUTPUT as an input "
                     "(Add Input -> Notebook Outputs); it only has them if it was saved with Save & Run All. Otherwise re-run notebook 04.")
print("fine-tuned models found:", sorted(os.listdir("models")))

## 3 · set the book's data aside, fetch and parse the scikit-learn pages (about 10 min)

In [ ]:
if not os.path.isdir("data_book"): shutil.move("data", "data_book"); os.mkdir("data")
!python scripts/fetch_sklearn_docs.py
!python -m gatekeep.cli prepare_dir data/sk

## 4 · load everything and start the app

In [ ]:
import json
from gatekeep import demo
from gatekeep.cli import load_chunks
from gatekeep.corpus import Index
from gatekeep.gates import LayaGate
from gatekeep.llm import LLM

llm, index = LLM(), Index(load_chunks())
laya = LayaGate.load(json.load(open("models/ckpts.json")))
taus = json.load(open("results/tau.json"))
demo.launch(index, llm, laya, taus, examples=[
    "What is the difference between bagging and boosting?",
    "How does k-means choose the number of clusters?",
    "How do I fine-tune a large language model with LoRA?",
    "hello, thanks for the help!",
])